In [ ]:
import pandas as pd
import numpy as np

# calculate the relative positions and velocities of surrounding vehicles
def calculate_relative_positions_and_velocities(tracks_df, ego_id=5):
    def calculate_relative_data(row):
        # Get IDs of surrounding vehicles in a specific order
        surrounding_ids = [
            row['leftPrecedingId'], row['precedingId'], row['rightPrecedingId'],
            row['leftAlongsideId'], row['id'], row['rightAlongsideId'],
            row['leftFollowingId'], row['followingId'], row['rightFollowingId']
        ]

        surrounding_data = {}
        for i, s_id in enumerate(surrounding_ids):
            if s_id > 0:
                s_vehicle = tracks_df[tracks_df['id'] == s_id]
                if not s_vehicle.empty:
                    # Calculate relative positions and velocities
                    dx = s_vehicle['x'].values[0] - row['x']
                    dy = s_vehicle['y'].values[0] - row['y']
                    dvx = s_vehicle['xVelocity'].values[0] - row['xVelocity']
                    dvy = s_vehicle['yVelocity'].values[0] - row['yVelocity']
                    surrounding_data[f'vehicle_{i+1}_relative_x'] = dx
                    surrounding_data[f'vehicle_{i+1}_relative_y'] = dy
                    surrounding_data[f'vehicle_{i+1}_relative_vx'] = dvx
                    surrounding_data[f'vehicle_{i+1}_relative_vy'] = dvy
                else:
                    # If no vehicle, set default values
                    surrounding_data[f'vehicle_{i+1}_relative_x'] = -1
                    surrounding_data[f'vehicle_{i+1}_relative_y'] = -1
                    surrounding_data[f'vehicle_{i+1}_relative_vx'] = 0
                    surrounding_data[f'vehicle_{i+1}_relative_vy'] = 0
            else:
                surrounding_data[f'vehicle_{i+1}_relative_x'] = -1
                surrounding_data[f'vehicle_{i+1}_relative_y'] = -1
                surrounding_data[f'vehicle_{i+1}_relative_vx'] = 0
                surrounding_data[f'vehicle_{i+1}_relative_vy'] = 0

        # Add ego vehicle values
        surrounding_data['vehicle_5_relative_x'] = row['x']
        surrounding_data['vehicle_5_relative_y'] = row['y']
        surrounding_data['vehicle_5_relative_vx'] = row['xVelocity']
        surrounding_data['vehicle_5_relative_vy'] = row['yVelocity']

        return surrounding_data

    relative_data = tracks_df.apply(calculate_relative_data, axis=1)
    relative_data_df = pd.DataFrame(relative_data.tolist())
    tracks_df = pd.concat([tracks_df, relative_data_df], axis=1)
    return tracks_df

# Function to convert the data into a matrix format
def reshape_to_vehicle_matrix(tracks_df):
    matrix_data = []
    for _, row in tracks_df.iterrows():
        vehicle_data = []
        for vehicle_num in range(1, 10):
            vehicle_features = [
                row.get(f"vehicle_{vehicle_num}_relative_x", -1),
                row.get(f"vehicle_{vehicle_num}_relative_y", -1),
                row.get(f"vehicle_{vehicle_num}_relative_vx", 0),
                row.get(f"vehicle_{vehicle_num}_relative_vy", 0),
            ]
            vehicle_data.append(vehicle_features)

        # Flatten the data into a single row
        matrix_data.append(np.array(vehicle_data).flatten())

    matrix_columns = [f"feature_{i+1}" for i in range(len(matrix_data[0]))]
    vehicle_matrix = pd.DataFrame(matrix_data, columns=matrix_columns)
    return vehicle_matrix

# Function to fill missing values
def fill_missing_values(df):
    position_columns = [col for col in df.columns if 'relative_x' in col or 'relative_y' in col]
    velocity_columns = [col for col in df.columns if 'relative_vx' in col or 'relative_vy' in col]

    ego_vx = df['xVelocity']
    ego_vy = df['yVelocity']

    for col in position_columns:
        df[col] = df[col].fillna(-1)

    for col in velocity_columns:
        if 'relative_vx' in col:
            df[col] = df[col].fillna(ego_vx)
        elif 'relative_vy' in col:
            df[col] = df[col].fillna(ego_vy)

    return df

# Function to split data into time intervals
def assign_time_intervals(df, interval=75):
    df['time'] = 'T1'
    max_frame = df['frame'].max()
    time_labels = [f'T{i+1}' for i in range((max_frame // interval) + 1)]
    for i, label in enumerate(time_labels):
        start_frame = i * interval
        end_frame = start_frame + interval - 1
        df.loc[(df['frame'] >= start_frame) & (df['frame'] <= end_frame), 'time'] = label
    return df

# Main Execution
if __name__ == "__main__":
    # Load the track and meta files
    tracks_df = pd.read_csv(r"D:\SEM_2_STUDY\Project\jadhav newdatafor flipping\44_tracks.csv")
    meta_df = pd.read_csv(r"D:\SEM_2_STUDY\Project\highD-dataset-v1.0\data\44_tracksMeta.csv")

    # Merge track and meta files
    combined_df = tracks_df.merge(meta_df, on='id', how='left')

    # Calculate relative positions and velocities
    combined_df = calculate_relative_positions_and_velocities(combined_df)

    # Fill missing values in the data
    combined_df = fill_missing_values(combined_df)

    # Assign time intervals for splitting the data
    combined_df = assign_time_intervals(combined_df)

    # Convert the data into a matrix format
    vehicle_matrix = reshape_to_vehicle_matrix(combined_df)

    # Add the matrix to the original data
    combined_df['vehicle_matrix'] = vehicle_matrix.apply(lambda row: row.tolist(), axis=1)

    # Save the final processed data to a CSV file
    combined_df.to_csv(r'D:\SEM_2_STUDY\Project\Matrix_Final\Matrix_track_44.csv', index=False)

    print("Data processing complete. File saved as 'processed_traffic_data_with_matrix_and_time44.csv'.")
